# Submission — RF V1 (sans leakage)

**Pipeline propre** :
1. Charger le **train complet** (X_train_sample + y_train_sample)
2. Feature engineering
3. Fit scalers sur tout le train → transform le test
4. Entraîner RF → prédire sur test → submission

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

SEED = 42
print('Libraries loaded.')

## 1. Charger les best params

In [ ]:
with open('best_params_rf.json', 'r') as f:
    best_params = json.load(f)
print('Best params RF:', best_params)

## 2. Feature Engineering (même fonction que V2)

In [ ]:
def feature_engineering(df):
    """Feature engineering par ligne (identique à V2)."""
    data = df.copy()
    ret_cols = [f'RET_{i}' for i in range(1, 21)]
    
    data['RET_1_TURNOVER'] = data['RET_1'] * data['MEDIAN_DAILY_TURNOVER']
    data['RET_MEAN_5'] = data[[f'RET_{i}' for i in range(1, 6)]].mean(axis=1)
    data['RET_MEAN_20'] = data[ret_cols].mean(axis=1)
    data['RET_STD_5'] = data[[f'RET_{i}' for i in range(1, 6)]].std(axis=1)
    data['RET_STD_20'] = data[ret_cols].std(axis=1)
    data['RET_CUM_5'] = data[[f'RET_{i}' for i in range(1, 6)]].sum(axis=1)
    data['RET_CUM_20'] = data[ret_cols].sum(axis=1)
    data['RET_POSITIVE_COUNT_5'] = (data[[f'RET_{i}' for i in range(1, 6)]] > 0).sum(axis=1)
    data['RET_POSITIVE_COUNT_20'] = (data[ret_cols] > 0).sum(axis=1)
    data['RET_RECENT_VS_OLD'] = data['RET_MEAN_5'] - data[[f'RET_{i}' for i in range(16, 21)]].mean(axis=1)
    
    return data

print('Feature engineering function defined.')

## 3. Préparer le Train

Fit les scalers sur **tout le train** (sans split, c'est pour la submission finale).

In [ ]:
# Charger train
X_train_raw = pd.read_csv('Data/X_train_sample.csv')
y_train_raw = pd.read_csv('Data/y_train_sample.csv')
df_train = X_train_raw.merge(y_train_raw, on='ROW_ID')

# Feature engineering
df_train = feature_engineering(df_train)
df_train['target_SIGN'] = (df_train['target'] > 0).astype(int)

# Colonnes features
exclude_cols = ['ROW_ID', 'TS', 'ALLOCATION', 'target', 'target_SIGN']
feature_cols = [c for c in df_train.columns if c not in exclude_cols]

X_train = df_train[feature_cols].fillna(0)
y_train = df_train['target_SIGN'].values

# Normalisation par groupe — FIT sur tout le train
numeric_cols = [c for c in feature_cols if c != 'GROUP']
train_scalers = {}
for grp in X_train['GROUP'].unique():
    mask = X_train['GROUP'] == grp
    scaler = StandardScaler()
    X_train.loc[mask, numeric_cols] = scaler.fit_transform(X_train.loc[mask, numeric_cols])
    train_scalers[grp] = scaler

print(f'Train : {X_train.shape}')
print(f'Features : {len(feature_cols)}')
print(f'Scalers fit sur {len(train_scalers)} groupes')

## 4. Préparer le Test

Même FE, puis **transform** avec les scalers du train (PAS de fit).

In [ ]:
# Charger test
X_test_raw = pd.read_csv('Data/X_test.csv')
print(f'Test brut : {X_test_raw.shape}')

# Feature engineering (même que train)
X_test_fe = feature_engineering(X_test_raw)

# Garder les ROW_ID pour la submission
test_row_ids = X_test_fe['ROW_ID']

# Sélectionner les mêmes features
X_test = X_test_fe[feature_cols].fillna(0)

# Normalisation — TRANSFORM avec les scalers du train
for grp in X_test['GROUP'].unique():
    mask = X_test['GROUP'] == grp
    if grp in train_scalers:
        X_test.loc[mask, numeric_cols] = train_scalers[grp].transform(X_test.loc[mask, numeric_cols])
    else:
        print(f'  ⚠️ Groupe {grp} inconnu dans le train → fit_transform')
        scaler = StandardScaler()
        X_test.loc[mask, numeric_cols] = scaler.fit_transform(X_test.loc[mask, numeric_cols])

print(f'Test normalisé : {X_test.shape}')

## 5. Entraîner et Prédire

In [ ]:
print('Training Random Forest...')
rf = RandomForestClassifier(**best_params, random_state=SEED, n_jobs=-1)
rf.fit(X_train.values, y_train)

# Prédire sur le test
y_pred = rf.predict(X_test.values)

print(f'Prédictions :')
print(f'  Up (1)   : {(y_pred == 1).sum()} ({(y_pred == 1).mean()*100:.1f}%)')
print(f'  Down (0) : {(y_pred == 0).sum()} ({(y_pred == 0).mean()*100:.1f}%)')

## 6. Créer le fichier de submission

In [ ]:
submission = pd.DataFrame({
    'ROW_ID': test_row_ids,
    'prediction': y_pred
})

submission.to_csv('submission_rf_V1.csv', index=False)

# Vérification format
sample = pd.read_csv('sample_submission.csv')
assert list(submission.columns) == list(sample.columns), f'Colonnes: {submission.columns.tolist()} vs {sample.columns.tolist()}'
assert len(submission) == len(sample), f'Taille: {len(submission)} vs {len(sample)}'

print(f'✅ submission_rf_V1.csv sauvegardé')
print(f'   {submission.shape[0]:,} lignes')
print(f'   Format vérifié vs sample_submission.csv')
print(f'\nAperçu :')
print(submission.head(10))